#1. Managed Table (Default):
Databricks manages both data and metadata.

When to use:
When you want Databricks to fully manage the table lifecycle.

Drop Effect:<br>
DROP TABLE → deletes both data and metadata.

#2. External Table:
Stores metadata in metastore but data in an external location (e.g., DBFS, S3, ADLS, GCS).

When to use:
When the data already exists outside Databricks and you just want to reference it.

Drop Effect:
DROP TABLE → removes only metadata, data remains.

#3. Temporary View:

A session-scoped logical view (not saved in the metastore).

Available only in the current Spark session.
#4. Global Temporary View:

A cross-session temporary view stored in the global_temp database.

#5. Delta Table:

A table stored in Delta Lake format, supporting ACID transactions, time travel, and schema enforcement.
Often used as both managed or external tables.

#Summary:
| Table Type       | Stored In Metastore | Data Location Controlled By | Persistent Across Sessions | Drop Deletes Data          |
| ---------------- | ------------------- | --------------------------- | -------------------------- | -------------------------- |
| Managed Table    | ✅ Yes               | Databricks                  | ✅ Yes                      | ✅ Yes                      |
| External Table   | ✅ Yes               | User                        | ✅ Yes                      | ❌ No                       |
| Temporary View   | ❌ No                | In-memory                   | ❌ No                       | 🕐 Ends with session       |
| Global Temp View | ✅ (global_temp db)  | In-memory                   | ✅ Across sessions          | 🕐 Ends with cluster       |
| Delta Table      | ✅ (if created)      | User or Databricks          | ✅ Yes                      | Depends (managed/external) |



#Syntax:
CREATE [OR REPLACE] [TEMPORARY | EXTERNAL] TABLE [IF NOT EXISTS] table_name
(
    column_name1 data_type [COMMENT col_comment],
    column_name2 data_type,
    ...
)
[USING file_format]
[OPTIONS (key1 = value1, key2 = value2, ...)]
[LOCATION 'path']
[TBLPROPERTIES (key1 = value1, key2 = value2, ...)]
[AS SELECT * FROM another_table];


#Key Clauses Explained
| Clause                   | Description                                                           |
| ------------------------ | --------------------------------------------------------------------- |
| **CREATE TABLE**         | Defines a new table.                                                  |
| **OR REPLACE**           | Replaces an existing table with the same name.                        |
| **TEMPORARY / EXTERNAL** | Creates a temporary (session-based) or external (user-managed) table. |
| **IF NOT EXISTS**        | Prevents error if the table already exists.                           |
| **USING**                | Specifies the file format (e.g., `parquet`, `csv`, `delta`, `json`).  |
| **OPTIONS**              | Used for format-specific options (like header, delimiter, etc.).      |
| **LOCATION**             | Defines where data physically resides (used for external tables).     |
| **TBLPROPERTIES**        | Adds metadata properties to the table.                                |
| **AS SELECT**            | Creates a table from the result of a query.                           |


In [0]:

%sql
-- Create temp view using SQL query
CREATE OR REPLACE TEMP VIEW temp_view
USING CSV
OPTIONS (
  inferSchema "true",
  header "true",
  path "dbfs:/Volumes/spark_workouts/customerdb/customerfile/cusdata/"
);

SELECT * FROM temp_view;

CREATE TABLE databricks_practice.outputdb.tblcustomers
USING DELTA
AS SELECT *
FROM temp_view;




In [0]:
%sql
CREATE VIEW databricks_practice.outputdb.tblcustomers_vw
SCHEDULE EVERY 1 hour
AS SELECT *
FROM databricks_practice.outputdb.tblcustomers;

In [0]:
%sql
USE CATALOG databricks_practice;
USE SCHEMA outputdb;

CREATE OR REPLACE MATERIALIZED VIEW databricks_practice.outputdb.tblcustomers_mvw
AS
SELECT profession,
       AVG(age) AS avg_age,
       COUNT(custid) AS cust_count
FROM databricks_practice.outputdb.tblcustomers
GROUP BY profession;


In [0]:
%sql
show databases;
show tables from databricks_practice.outputdb;
show columns in databricks_practice.outputdb;